In [1]:
import os
import torch
import fitz  # PyMuPDF
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew, Process
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from crewai.tools import tool
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load environment variables
load_dotenv()

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

# Set dummy key for CrewAI internal validation
os.environ["OPENAI_API_KEY"] = "NA"

# Initialize LLMs
legal_llm = LLM(model="groq/llama-3.3-70b-versatile")
research_llm = LLM(model="groq/llama-3.1-8b-instant")

✅ Using device: cuda


In [2]:
import os
# This forces the agent to use text-based thought processes instead of native JSON tools
os.environ["CREWAI_USE_NATIVE_TOOLS"] = "false"

In [3]:
# 1. Initialize Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={'device': device}
)

# 2. Connect to existing Vector Database
db_path = "C:/Users/user/Desktop/RAG Projects/Legal RAG 1/vector_database"
vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)
print(f"✅ Connected to Vector Store: {vector_store._collection.count()} chunks found.")

# 3. Manual Summarizer Logic (Bypasses Pipeline KeyError)
model_id = "facebook/bart-large-cnn"
print("⏳ Loading Summarizer components...")
summary_tokenizer = AutoTokenizer.from_pretrained(model_id)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

def compress_legal_text(text):
    """Summarizes legal text using direct model inference to avoid registry errors."""
    if len(text) < 2000: return text
    input_text = text[:7000] # Focus on facts/judgment
    inputs = summary_tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    summary_ids = summary_model.generate(inputs["input_ids"], num_beams=4, min_length=200, max_length=600)
    return summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Connected to Vector Store: 0 chunks found.
⏳ Loading Summarizer components...


Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [4]:
@tool("constitutional_search_tool")
def constitutional_search_tool(query: str):
    """
    Mandatory: Use this tool to search the Indian Constitution.
    The input MUST be a simple string. Do not provide JSON or complex objects.
    Example query: 'Article 21'
    """
    docs = vector_store.similarity_search(query, k=3)
    return "\n\n".join([f"Law Content: {doc.page_content}" for doc in docs])

@tool("document_analyzer_tool")
def document_analyzer_tool(file_path: str):
    """
    Reads and summarizes an uploaded PDF document. 
    Use this ONLY if a valid file path is provided. 
    If file_path is 'None' or empty, tell the user no document was provided.
    """
    # Clean the input
    if not file_path or str(file_path).lower() in ["none", "", "null"]:
        return "System Note: No document was provided by the user for this specific query. Please proceed using only the Constitutional RAG or general knowledge."

    try:
        # Check if it's a URL (like in your error log) or a local path
        if file_path.startswith("http"):
            return "Error: This tool only supports local PDF files, not Google Doc links. Please download the PDF to your local 'data' folder."

        doc = fitz.open(file_path)
        text = "".join([page.get_text() for page in doc])
        return compress_legal_text(text)
    except Exception as e:
        return f"Error accessing document: {str(e)}"

In [5]:
def create_legal_crew(pdf_path="None"):
    # Ensure pdf_path is passed as a string
    path_str = str(pdf_path)

    context_agent = Agent(
        role='Legal Context Provider',
        goal='Identify facts from the provided document or define the scope of the standalone query.',
        backstory='You are a specialist in document review. If a file path is provided, you MUST use the document_analyzer_tool.',
        tools=[document_analyzer_tool], # Ensure it is HERE
        llm=research_llm,
        verbose=True,
        allow_delegation=False
    )

    # Agent 2: The Scholar (RAG Specialist)
    scholar_agent = Agent(
        role='Constitutional Scholar',
        goal='Map the query or case facts to the Indian Constitution.',
        backstory='Expert in Indian Constitutional Law and legal precedents.',
        tools=[constitutional_search_tool],
        llm=research_llm,
        verbose=True
    )

    # Agent 3: The Advisor
    advisor_agent = Agent(
        role='Legal Advisor',
        goal='Provide a final, comprehensive answer to the user.',
        backstory='Senior Counsel who merges case facts with supreme law.',
        llm=legal_llm,
        verbose=True
    )

    # Add this to each Agent definition in Cell 5
    context_agent = Agent(
        role='...',
        goal='...',
        backstory='...',
        tools=[document_analyzer_tool],
        llm=research_llm,
        verbose=True,
        allow_delegation=False,
        max_iter=3,  # Prevents infinite loops if tool fails
        handle_parsing_errors=True # Crucial: tells the agent how to recover from JSON errors
    )

    # Define Tasks
    t1 = Task(
        description=f"""
        1. Check if the document path is valid: {path_str}.
        2. If valid, use the document_analyzer_tool to extract facts.
        3. If the path is 'None', analyze the standalone query: {{user_query}}
        """,
        expected_output="A summary of the case facts or a legal scope definition.",
        agent=context_agent
    )

    t2 = Task(
        description="Search for applicable Constitutional Articles based on the facts provided.",
        expected_output="A list of relevant laws and sections.",
        agent=scholar_agent,
        context=[t1]
    )

    t3 = Task(
        description="Generate the final response for: {{user_query}}. Be precise and professional.",
        expected_output="The final legal advice.",
        agent=advisor_agent,
        context=[t2]
    )

    return Crew(
        agents=[context_agent, scholar_agent, advisor_agent],
        tasks=[t1, t2, t3],
        process=Process.sequential,
        max_rpm=2
    )

In [6]:
def run_legal_system(query, pdf_file="None"):
    print(f"🚀 Processing... Mode: {'Hybrid' if pdf_file != 'None' else 'Standalone'}")
    crew = create_legal_crew(pdf_file)
    result = crew.kickoff(inputs={"user_query": query})
    
    print("\n" + "="*50)
    print("LEGAL RESPONSE")
    print("="*50 + "\n")
    print(result.raw)

In [7]:
# --- EXAMPLE 1: STANDALONE QUESTION ---
run_legal_system("If a state law conflicts with a central law in India, which one prevails?")

🚀 Processing... Mode: Standalone


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: ...                                                                                                     │
│                                                                                                                 │
│  Task:                                                                                                          │
│          1. Check if the document path is valid: None.                                                          │
│          2. If valid, use the document_analyzer_tool to extract facts.                                          │
│          3. If the path is 'None', analyze the standalone query: If a state law conflicts with a central law    │
│  in India, which one prevails?                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

BadRequestError: litellm.BadRequestError: GroqException - {"error":{"message":"Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"\u003cfunction=document_analyzer_tool\u003e{\"file_path\": None}\u003c/function\u003e\n\n"}}
